In [ ]:
# imports

import pandas as pd
import numpy as np
import json
import polars as pl
from pathlib import Path
# There is also auto tokenizer for general purpose but just using BERT for simplicity
from datasets import Dataset, DatasetDict, concatenate_datasets
import torch
from torch.utils.data import DataLoader
import glob
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (set_seed,
                          TrainingArguments,
                          Trainer,
                          Qwen3ForCausalLM,
                          AutoTokenizer,
                          DataCollatorWithPadding)
from tqdm import tqdm

In [ ]:
CURRENT_SEED = 42

set_seed(CURRENT_SEED)

In [ ]:
# Extract tabfact zip if notebook is in google colab

import zipfile
import os

zip_path = '/content/tabfact_data.zip'
extract_path = '/content/'

print("Unzipping dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipping completed! Files extracted to:", extract_path)

In [ ]:
# Set data path, if notebook is in google colab use content, if not it is in the directory above

# # Local repository path
# DATA_PATH = "../tabfact_data"

# Colab path
DATA_PATH = "/content/tabfact_data"

In [ ]:
# Helpers to read in the json data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, label, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, label in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "label": label,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [ ]:
# Load in table dataframe

# DOESN'T WORK IN COLAB BUT FASTER IN LOCAL REPO
df_tables = (
    pl.read_csv(
        "../tabfact_data/tables/*.csv",
        has_header=False,
        separator="\1", # Non-existent delimiter so full line/file stays intact
        new_columns=["table_content"],
        include_file_paths="file_name",
    )
    .group_by("file_name", maintain_order=True) # Group all the table lines together 
    .agg(pl.col("table_content").str.join("\n")) # Concatenate table lines together
    .to_pandas()
)

# csv_files = sorted(glob.glob(f"{DATA_PATH}/tables/*.csv"))

# df_tables = (
#     pl.concat([
#         pl.read_csv(
#             file,
#             has_header=False,
#             separator="\1",
#             new_columns=["table_content"],
#         ).with_columns(pl.lit(Path(file).name).alias("file_name"))
#         for file in csv_files
#     ])
#     .group_by("file_name", maintain_order=True)  # Group all the table lines together
#     .agg(pl.col("table_content").str.join("\n"))  # Concatenate table lines together
#     .to_pandas()
# )

df_tables["file_name"] = df_tables["file_name"].apply(lambda p: Path(p).name) # Only keep the table id

In [ ]:
# Defining column labelled rows
print(df_tables.shape)

df_tables["headers"] = df_tables["table_content"].apply(lambda x: x.split("\n")[0].split("#")) # Get the first line of the table content as headers

df_tables["rows"] = df_tables["table_content"].apply(lambda x: [row.split("#") for row in x.split("\n")[1:]]) # Get the rest of the lines as rows

df_tables["column_labelled_rows"] = df_tables.apply(
    lambda row: "\n".join(
        ", ".join(f"{h}: {v}" for h, v in zip(row["headers"], r))
        for r in row["rows"]
    ),
    axis=1,
)

print(df_tables["column_labelled_rows"].iloc[0])

In [ ]:
# Load in non table data frames

df_full_cleaned = load_and_convert_data_to_dataframe(f"{DATA_PATH}/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['label'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['label'] == 1]

df_r1_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r1_training_all.json")
df_r2_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe(f"{DATA_PATH}/train_examples.json")
df_test = load_and_convert_data_to_dataframe(f"{DATA_PATH}/test_examples.json")
df_validation = load_and_convert_data_to_dataframe(f"{DATA_PATH}/val_examples.json")


In [ ]:
NO_THINKING_PROMPT = "You are a binary classifier. Respond only with 1 or 0. If the CLAIM is entailed by the TABLE, respond with 1. If the CLAIM is refuted by the table respond with 0."
THINKING_PROMPT = "You are a binary classifier. If the CLAIM is entailed by the TABLE, respond with 1. If the CLAIM is refuted by the TABLE respond with 0. Reason carefully inside <think></think> before deciding. After </think>, output ONLY 1 or 0 with no extra text. "

In [ ]:
def get_table_claim_df(df, table_df_col, table_before_claim=True, table_prefix="TABLE: ", claim_prefix="CLAIM: ", table_claim_separator=" | ", before_evidence_text=""):
    df_table_claim = pd.merge(
        df_tables[["file_name", table_df_col]],
        df[["table_id", "statement", "label"]],
        left_on="file_name",
        right_on="table_id",
        how="inner"
    ).reset_index(drop=True)
    if table_before_claim:
      before_evidence_text += "Here is the TABLE and then the CLAIM: \n" if before_evidence_text != "" else ""
      df_table_claim["table_claim"] = (
        before_evidence_text +
        table_prefix +
        df_table_claim[table_df_col] +
        table_claim_separator +
        claim_prefix +
        df_table_claim["statement"]
      )
    else:
        before_evidence_text += "Here is the CLAIM and then the TABLE: \n" if before_evidence_text != "" else ""
        df_table_claim["table_claim"] = (
          before_evidence_text +
          claim_prefix +
          df_table_claim["statement"] +
          table_claim_separator +
          table_prefix +
          df_table_claim[table_df_col]
        )

    return df_table_claim[["table_claim", "label"]].copy()

In [ ]:
df_train_table_claim = get_table_claim_df(df_training, "table_content", table_before_claim=False)
df_val_table_claim = get_table_claim_df(df_validation, "table_content", table_before_claim=False, table_claim_separator="\n")
df_test_table_claim = get_table_claim_df(df_test, "table_content", table_before_claim=False, table_claim_separator="\n")
df_train_r1_table_claim = get_table_claim_df(df_r1_training, "table_content", table_before_claim=False)
df_train_r2_table_claim = get_table_claim_df(df_r2_training, "table_content", table_before_claim=False)

print(df_val_table_claim["table_claim"].iloc[0])

print(df_val_table_claim.shape)

In [ ]:
# defining qwen model and tokenizer 
qwen_model = Qwen3ForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    device_map="auto",
    torch_dtype=torch.float32,
)

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
qwen_tokenizer.padding_side = "left"
qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qwen_model.to(device)
qwen_model.eval()

In [ ]:
def get_reduced_random_test_dataset(size):
  entailed_test_dataset = Dataset.from_pandas(df_test_table_claim[df_test_table_claim['label'] == 1])
  refuted_test_dataset = Dataset.from_pandas(df_test_table_claim[df_test_table_claim['label'] == 0])

  generator = np.random.default_rng(CURRENT_SEED)
  entailed_random_indices = generator.choice(len(entailed_test_dataset), size=int(size/2), replace=False)
  refuted_random_indices = generator.choice(len(refuted_test_dataset), size=int(size/2), replace=False)

  return concatenate_datasets([
      entailed_test_dataset.select(entailed_random_indices),
      refuted_test_dataset.select(refuted_random_indices)
  ])

In [ ]:
r1_entailed = df_train_r1_table_claim[df_train_r1_table_claim["label"] == 1]
r1_refuted = df_train_r1_table_claim[df_train_r1_table_claim["label"] == 0]
r2_entailed = df_train_r2_table_claim[df_train_r2_table_claim["label"] == 1]
r2_refuted = df_train_r2_table_claim[df_train_r2_table_claim["label"] == 0]

In [ ]:
def df_entry_to_shot(df_entry):
  return [
      {
        "role": "user",
        "content": df_entry["table_claim"]
      },
      {
        "role": "assistant",
        "content": df_entry["label"]
      }]

def get_shots(is_from_r1_list):
  shots = []
  r1_shots_used = 0
  r2_shots_used = 0
  for i in range(len(is_from_r1_list)):
    is_r1 = is_from_r1_list
    get_entailed_shot = i % 2 == 0

    if is_r1:
      if get_entailed_shot:
        df_entry = r1_entailed.iloc[r1_shots_used]
      else:
        df_entry = r1_refuted.iloc[r1_shots_used]
      r1_shots_used += 1

    else:
      if get_entailed_shot:
        df_entry = r2_entailed.iloc[r2_shots_used]
      else:
        df_entry = r2_refuted.iloc[r2_shots_used]
      r2_shots_used += 1

    shots += df_entry_to_shot(df_entry)
  return shots

get_shots([True, True])

In [ ]:
# Preprocess function that correctly applies the chat template to each entry and returns the tokenized batch
def preprocess_batch(entries, thinking=True, shots=0):
    formatted_prompts = []
    for table_claim in entries["table_claim"]:
        prompt = THINKING_PROMPT if thinking else NO_THINKING_PROMPT
        messages = [
            {
                "role": "system",
                "content": prompt
            }]

        if shots > 0:
          messages += get_shots(shots)

        messages.append({
            "role": "user",
            "content": table_claim
        })

        # Use apply_chat_template with add_generation_prompt=True to format as raw text for generation
        prompt_text = qwen_tokenizer.apply_chat_template(messages,
                                                         tokenize=False,
                                                         add_generation_prompt=True,
                                                         enable_thinking=thinking)
        formatted_prompts.append(prompt_text)

    return qwen_tokenizer(formatted_prompts, truncation=True, return_attention_mask=True)

test_dataset = Dataset.from_pandas(df_test_table_claim)

datasets = DatasetDict({
    "test": test_dataset,
})

tokenized_datasets = datasets.map(preprocess_batch(thinking=False), batched=True, remove_columns="table_claim")

In [ ]:
print(pd.Series(tokenized_datasets["test"]["label"]).value_counts())

In [ ]:
# Compute token length for each sample in the dataset
# Assume tokenized_datasets["test"] contains 'input_ids' without padding
lengths = [len(ids) for ids in tokenized_datasets["test"]["input_ids"]]

min_len = int(np.min(lengths))
avg_len = float(np.mean(lengths))
median_len = int(np.median(lengths))
p95_len = int(np.percentile(lengths, 95))
max_len = int(np.max(lengths))

print(f"--- Token Length Statistics ({len(lengths)} samples) ---")
print(f"Min Length:     {min_len} tokens")
print(f"Average Length: {avg_len:.2f} tokens")
print(f"Median Length:  {median_len} tokens")
print(f"95th Percentile:{p95_len} tokens")
print(f"Max Length:     {max_len} tokens")

# Optional: Count how many samples exceed common context thresholds
over_1024 = sum(1 for l in lengths if l > 1024)
print(f"\nSamples exceeding 1024 tokens: {over_1024} ({(over_1024 / len(lengths)) * 100:.2f}%)")

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=qwen_tokenizer, padding=True)

test_loader = DataLoader(
    tokenized_datasets["test"],
    batch_size=8,
    collate_fn=data_collator,
    shuffle=False
)

In [ ]:
# Running zero/fewshot
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

predictions = []

labels = tokenized_datasets["test"]["label"]

with torch.no_grad():
  for batch in tqdm(test_loader, desc="Qwen3 Zero-Shot"):
          # get tokenized data
          input_ids = batch["input_ids"].to(qwen_model.device)
          attention_mask = batch["attention_mask"].to(qwen_model.device)

          # get predictions
          outputs = qwen_model.generate(
              input_ids=input_ids,
              attention_mask=attention_mask,
              max_new_tokens=512,
              pad_token_id=qwen_tokenizer.pad_token_id,
              do_sample=False,
          )

          # get only the tokens that got generated (concatenated to prompt)
          generated_tokens = outputs[:, input_ids.shape[1]:]
          decoded_batch = qwen_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
          predictions.extend(output.strip() for output in decoded_batch)

In [ ]:
#Output zero/fewshot results
failed = []
def clean_prediction(pred, thinking=True):
    # Get 1 or 0 integer from the qwen output
    if thinking:
        pred_str = pred.strip()[-1]
    else:
        pred_str = str(pred).strip().lower()

    if '1' in pred_str:
        return 1
    elif '0' in pred_str:
        return 0

    # Fallback when 1 or 0 wasn't given in the output
    failed.append(pred)
    return -1
failed = []

processed_predictions = [clean_prediction(p) for p in predictions]

# Filter out failures
valid_predictions = [(p, l) for p, l in zip(processed_predictions, labels) if p != -1]
invalid_predictions = [(p, l) for p, l in zip(processed_predictions, labels) if p == -1]
if len(valid_predictions) > 0:
    filtered_preds, filtered_labels = zip(*valid_predictions)

    # Calculate standard binary classification metrics
    accuracy = accuracy_score(filtered_labels, filtered_preds)
    precision = precision_score(filtered_labels, filtered_preds)
    recall = recall_score(filtered_labels, filtered_preds)
    f1 = f1_score(filtered_labels, filtered_preds)
else:
    accuracy = precision = recall = f1 = 0.0

failed_count = processed_predictions.count(-1)

print(f"Number of failed predictions: {failed_count} out of {len(processed_predictions)} ({failed_count/len(processed_predictions)*100:.2f}%)")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

In [ ]:
print(failed[0])